In [1]:
import os
import re
import json
from collections import defaultdict

import dashscope
from pymilvus import MilvusClient
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ============================================================
# 1. 基础配置
# ============================================================

if not os.getenv("DASHSCOPE_API_KEY"):
    raise RuntimeError(
        "没有找到 DASHSCOPE_API_KEY，请先在终端配置环境变量。"
    )

dashscope.api_key = os.environ["DASHSCOPE_API_KEY"]

LLM_MODEL = "glm-5.2"
EMBEDDING_MODEL = "qwen3.7-text-embedding"

MILVUS_DB_PATH = "./multi_query_demo.db"
COLLECTION_NAME = "multi_query_demo"


# ============================================================
# 2. 准备演示知识库
# ============================================================

documents = [
    {
        "source": "Python性能优化指南.txt",
        "text": """
        Python程序运行速度较慢时，可以首先使用cProfile定位性能瓶颈。
        对于计算密集型任务，可以使用NumPy向量化、Numba即时编译，
        或者将关键计算逻辑改写为C扩展。不要在没有性能分析数据的情况下
        盲目优化代码。
        """,
    },
    {
        "source": "程序并发处理指南.txt",
        "text": """
        对于网络请求、文件读取和数据库访问等I/O密集型任务，
        可以使用asyncio异步编程或者线程池提升吞吐量。
        对于CPU密集型任务，可以使用多进程绕过Python的GIL限制。
        """,
    },
    {
        "source": "缓存技术实践.txt",
        "text": """
        对于需要重复执行的高成本函数，可以使用缓存避免重复计算。
        Python中的functools.lru_cache适合缓存函数调用结果。
        分布式系统可以使用Redis保存跨进程共享的缓存数据。
        """,
    },
    {
        "source": "数据库访问优化.txt",
        "text": """
        数据库查询效率可以通过创建索引、减少全表扫描、批量写入和连接池
        等方式提升。应避免在循环中反复执行单条SQL查询，这种问题通常
        被称为N+1查询问题。
        """,
    },
    {
        "source": "JSON数据处理指南.txt",
        "text": """
        Python可以使用标准库中的json模块处理JSON数据。
        json.loads用于解析JSON字符串，json.load用于读取JSON文件，
        json.dumps用于把Python对象序列化为JSON字符串。
        """,
    },
    {
        "source": "代码质量规范.txt",
        "text": """
        提高代码可维护性需要使用清晰的命名、合理的函数划分、类型标注、
        单元测试和代码审查。代码可维护性与代码运行速度属于不同的质量属性。
        """,
    },
]


# ============================================================
# 3. 使用 LangChain 切分文档
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=30,
    separators=["\n\n", "\n", "。", "；", "，", " "],
)

chunks = []
chunk_id = 1

for document in documents:
    split_texts = text_splitter.split_text(document["text"].strip())

    for chunk_index, text in enumerate(split_texts):
        chunks.append(
            {
                "id": chunk_id,
                "text": text,
                "source": document["source"],
                "chunk_index": chunk_index,
            }
        )
        chunk_id += 1

print(f"原始文档数量：{len(documents)}")
print(f"切分后的 Chunk 数量：{len(chunks)}")

/opt/anaconda3/envs/agent-base/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


原始文档数量：6
切分后的 Chunk 数量：6


In [2]:
def embed_texts(
    texts: list[str],
    batch_size: int = 8,
) -> list[list[float]]:
    """
    使用 qwen3.7-text-embedding 批量生成向量。

    DashScope 返回结果中包含 text_index，
    所以要根据 text_index 排序，保证向量和原始文本一一对应。
    """
    if not texts:
        return []

    all_vectors = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        response = dashscope.TextEmbedding.call(
            model=EMBEDDING_MODEL,
            input=batch,
        )

        if response.status_code != 200:
            raise RuntimeError(
                f"Embedding调用失败："
                f"{response.code} - {response.message}"
            )

        embeddings = sorted(
            response.output["embeddings"],
            key=lambda item: item["text_index"],
        )

        all_vectors.extend(
            item["embedding"] for item in embeddings
        )

    if len(all_vectors) != len(texts):
        raise RuntimeError("Embedding数量和输入文本数量不一致")

    return all_vectors


# 动态检测向量维度，不把1024写死
probe_vector = embed_texts(["用于检测向量维度的文本"])[0]
EMBEDDING_DIMENSION = len(probe_vector)

print("Embedding模型：", EMBEDDING_MODEL)
print("向量维度：", EMBEDDING_DIMENSION)

Embedding模型： qwen3.7-text-embedding
向量维度： 1024


In [3]:
client = MilvusClient(uri=MILVUS_DB_PATH)

if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    dimension=EMBEDDING_DIMENSION,
    primary_field_name="id",
    id_type="int",
    vector_field_name="vector",
    metric_type="COSINE",
    auto_id=False,
    enable_dynamic_field=True,
)

chunk_vectors = embed_texts(
    [chunk["text"] for chunk in chunks]
)

entities = []

for chunk, vector in zip(chunks, chunk_vectors):
    entities.append(
        {
            "id": chunk["id"],
            "vector": vector,
            "text": chunk["text"],
            "source": chunk["source"],
            "chunk_index": chunk["chunk_index"],
        }
    )

insert_result = client.insert(
    collection_name=COLLECTION_NAME,
    data=entities,
)

print("写入结果：", insert_result)
print("Collection统计：", client.get_collection_stats(COLLECTION_NAME))

写入结果： {'insert_count': 6, 'ids': [1, 2, 3, 4, 5, 6]}
Collection统计： {'row_count': 6}


In [4]:
def parse_json_array(content: str) -> list[str]:
    """
    从模型返回内容中提取JSON数组。
    同时兼容模型偶尔返回Markdown代码块的情况。
    """
    content = content.strip()

    content = re.sub(
        r"^```(?:json)?\s*",
        "",
        content,
        flags=re.IGNORECASE,
    )
    content = re.sub(r"\s*```$", "", content)

    match = re.search(r"\[[\s\S]*\]", content)

    if not match:
        raise ValueError(f"未找到JSON数组，模型原始输出：{content}")

    result = json.loads(match.group())

    if not isinstance(result, list):
        raise TypeError("模型返回结果不是JSON数组")

    return [
        str(item).strip()
        for item in result
        if str(item).strip()
    ]


def generate_multi_queries(
    original_query: str,
    num_queries: int = 4,
) -> list[str]:
    """
    使用百炼Qwen为原始问题生成多个不同表达的查询。
    """
    prompt = f"""
你是一名企业知识库查询改写专家。

请把用户的原始问题改写为 {num_queries} 个查询。

要求：
1. 所有查询必须保持原始问题的真实意图；
2. 使用不同的关键词、表达方式和检索角度；
3. 不要回答原始问题；
4. 不要添加原始问题中不存在的限制条件；
5. 只返回一个合法的JSON字符串数组；
6. 不要输出Markdown代码块；
7. 不要输出解释。

原始问题：
{original_query}
""".strip()

    response = dashscope.Generation.call(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": "你负责生成用于知识库检索的多个查询。",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        result_format="message",
        temperature=0.7,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"LLM调用失败：{response.code} - {response.message}"
        )

    content = response.output["choices"][0]["message"]["content"]
    queries = parse_json_array(content)

    # 去重并限制数量
    unique_queries = []
    seen = set()

    for query in queries:
        normalized_query = query.strip().lower()

        if normalized_query not in seen:
            seen.add(normalized_query)
            unique_queries.append(query)

    if not unique_queries:
        raise RuntimeError("模型没有生成有效查询")

    return unique_queries[:num_queries]

In [5]:
original_query = "如何提高代码执行效率？"

generated_queries = generate_multi_queries(
    original_query=original_query,
    num_queries=4,
)

print("原始问题：")
print(original_query)

print("\nLLM生成的查询：")
for index, query in enumerate(generated_queries, start=1):
    print(f"Query {index}: {query}")

原始问题：
如何提高代码执行效率？

LLM生成的查询：
Query 1: 代码性能优化的常用方法有哪些？
Query 2: 如何降低程序运行时间并提升执行速度？
Query 3: 提升程序运行效率的算法与数据结构优化技巧
Query 4: 减少代码资源消耗以实现高性能的编码实践


I0901 16:56:15.334873 4756774 chttp2_transport.cc:1393] ipv4:127.0.0.1:60819: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0901 16:56:15.334915 4756774 chttp2_transport.cc:1425] ipv4:127.0.0.1:60819: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


In [6]:
def multi_query_search(
    original_query: str,
    num_queries: int = 4,
    top_k_per_query: int = 3,
    final_top_k: int = 5,
    include_original: bool = True,
):
    """
    Multi-query检索流程：

    1. 使用LLM生成多个查询；
    2. 为所有查询生成向量；
    3. 批量调用Milvus执行多路检索；
    4. 根据文档ID去重；
    5. 使用最高相似度重新排序。

    这里没有使用RRF，因此仍然属于普通Multi-query。
    """
    generated_queries = generate_multi_queries(
        original_query=original_query,
        num_queries=num_queries,
    )

    retrieval_queries = list(generated_queries)

    # 保留原始问题可以降低查询改写偏离原意的风险
    if include_original:
        retrieval_queries.insert(0, original_query)

    query_vectors = embed_texts(retrieval_queries)

    # data中传入多个查询向量，Milvus会分别执行检索
    search_results = client.search(
        collection_name=COLLECTION_NAME,
        data=query_vectors,
        limit=top_k_per_query,
        output_fields=[
            "text",
            "source",
            "chunk_index",
        ],
    )

    # 保存每一路查询的原始结果，方便观察
    results_by_query = {}

    # 按文档ID去重
    merged_results = {}

    for query, hits in zip(retrieval_queries, search_results):
        current_query_results = []

        for rank, hit in enumerate(hits, start=1):
            entity = hit.get("entity", {})
            document_id = hit["id"]
            score = float(hit["distance"])

            current_result = {
                "id": document_id,
                "rank": rank,
                "score": score,
                "source": entity.get("source"),
                "chunk_index": entity.get("chunk_index"),
                "text": entity.get("text"),
            }

            current_query_results.append(current_result)

            if document_id not in merged_results:
                merged_results[document_id] = {
                    "id": document_id,
                    "source": entity.get("source"),
                    "chunk_index": entity.get("chunk_index"),
                    "text": entity.get("text"),
                    "max_score": score,
                    "hit_count": 1,
                    "matched_queries": [query],
                }
            else:
                merged_results[document_id]["max_score"] = max(
                    merged_results[document_id]["max_score"],
                    score,
                )
                merged_results[document_id]["hit_count"] += 1
                merged_results[document_id]["matched_queries"].append(
                    query
                )

        results_by_query[query] = current_query_results

    # 普通Multi-query没有统一规定必须如何重排。
    # 这里先比较最高相似度，再比较命中次数。
    final_results = sorted(
        merged_results.values(),
        key=lambda item: (
            item["max_score"],
            item["hit_count"],
        ),
        reverse=True,
    )

    return {
        "original_query": original_query,
        "generated_queries": generated_queries,
        "retrieval_queries": retrieval_queries,
        "results_by_query": results_by_query,
        "final_results": final_results[:final_top_k],
    }

In [7]:
result = multi_query_search(
    original_query="如何提高代码执行效率？",
    num_queries=4,
    top_k_per_query=3,
    final_top_k=5,
    include_original=True,
)

I0901 17:14:04.050395 4756776 chttp2_transport.cc:1393] ipv4:127.0.0.1:60819: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E0901 17:14:04.050669 4756776 chttp2_transport.cc:1425] ipv4:127.0.0.1:60819: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 20000ms


In [8]:
for query, hits in result["results_by_query"].items():
    print("=" * 80)
    print("检索查询：", query)

    for hit in hits:
        print(
            f"排名={hit['rank']} | "
            f"相似度={hit['score']:.4f} | "
            f"来源={hit['source']}"
        )
        print(hit["text"])
        print()

检索查询： 如何提高代码执行效率？
排名=1 | 相似度=0.6546 | 来源=数据库访问优化.txt
数据库查询效率可以通过创建索引、减少全表扫描、批量写入和连接池
        等方式提升。应避免在循环中反复执行单条SQL查询，这种问题通常
        被称为N+1查询问题。

排名=2 | 相似度=0.5560 | 来源=Python性能优化指南.txt
Python程序运行速度较慢时，可以首先使用cProfile定位性能瓶颈。
        对于计算密集型任务，可以使用NumPy向量化、Numba即时编译，
        或者将关键计算逻辑改写为C扩展。不要在没有性能分析数据的情况下
        盲目优化代码。

排名=3 | 相似度=0.5405 | 来源=程序并发处理指南.txt
对于网络请求、文件读取和数据库访问等I/O密集型任务，
        可以使用asyncio异步编程或者线程池提升吞吐量。
        对于CPU密集型任务，可以使用多进程绕过Python的GIL限制。

检索查询： 优化程序性能的方法有哪些？
排名=1 | 相似度=0.6342 | 来源=Python性能优化指南.txt
Python程序运行速度较慢时，可以首先使用cProfile定位性能瓶颈。
        对于计算密集型任务，可以使用NumPy向量化、Numba即时编译，
        或者将关键计算逻辑改写为C扩展。不要在没有性能分析数据的情况下
        盲目优化代码。

排名=2 | 相似度=0.5759 | 来源=数据库访问优化.txt
数据库查询效率可以通过创建索引、减少全表扫描、批量写入和连接池
        等方式提升。应避免在循环中反复执行单条SQL查询，这种问题通常
        被称为N+1查询问题。

排名=3 | 相似度=0.5292 | 来源=程序并发处理指南.txt
对于网络请求、文件读取和数据库访问等I/O密集型任务，
        可以使用asyncio异步编程或者线程池提升吞吐量。
        对于CPU密集型任务，可以使用多进程绕过Python的GIL限制。

检索查询： 代码运行速度慢如何优化？
排名=1 | 相似度=0.6928 | 来源=Python性能优化指南.txt
Python程序运

In [9]:
print("\n最终合并结果")
print("=" * 80)

for rank, item in enumerate(result["final_results"], start=1):
    print(f"\n最终排名：{rank}")
    print(f"来源：{item['source']}")
    print(f"最高相似度：{item['max_score']:.4f}")
    print(f"被不同查询命中的次数：{item['hit_count']}")

    print("命中该文档的查询：")
    for query in item["matched_queries"]:
        print("  -", query)

    print("文档内容：")
    print(item["text"])


最终合并结果

最终排名：1
来源：Python性能优化指南.txt
最高相似度：0.6928
被不同查询命中的次数：4
命中该文档的查询：
  - 如何提高代码执行效率？
  - 优化程序性能的方法有哪些？
  - 代码运行速度慢如何优化？
  - 代码性能调优指南或最佳实践
文档内容：
Python程序运行速度较慢时，可以首先使用cProfile定位性能瓶颈。
        对于计算密集型任务，可以使用NumPy向量化、Numba即时编译，
        或者将关键计算逻辑改写为C扩展。不要在没有性能分析数据的情况下
        盲目优化代码。

最终排名：2
来源：数据库访问优化.txt
最高相似度：0.6546
被不同查询命中的次数：5
命中该文档的查询：
  - 如何提高代码执行效率？
  - 优化程序性能的方法有哪些？
  - 代码运行速度慢如何优化？
  - 代码性能调优指南或最佳实践
  - 提升算法执行效率和减少资源消耗的技巧
文档内容：
数据库查询效率可以通过创建索引、减少全表扫描、批量写入和连接池
        等方式提升。应避免在循环中反复执行单条SQL查询，这种问题通常
        被称为N+1查询问题。

最终排名：3
来源：缓存技术实践.txt
最高相似度：0.5833
被不同查询命中的次数：1
命中该文档的查询：
  - 提升算法执行效率和减少资源消耗的技巧
文档内容：
对于需要重复执行的高成本函数，可以使用缓存避免重复计算。
        Python中的functools.lru_cache适合缓存函数调用结果。
        分布式系统可以使用Redis保存跨进程共享的缓存数据。

最终排名：4
来源：程序并发处理指南.txt
最高相似度：0.5714
被不同查询命中的次数：4
命中该文档的查询：
  - 如何提高代码执行效率？
  - 优化程序性能的方法有哪些？
  - 代码性能调优指南或最佳实践
  - 提升算法执行效率和减少资源消耗的技巧
文档内容：
对于网络请求、文件读取和数据库访问等I/O密集型任务，
        可以使用asyncio异步编程或者线程池提升吞吐量。
        对于CPU密集型任务，可以使用多进程绕过Python的GIL限制。

最终排名：5
来源：代码质量规范.txt
最